# 08 - Thesis Evidence Synthesis Across IEEE-CIS and BAF

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def sync_kaggle_project() -> Path:
    """Use one current working clone; never import source bundled in an input artifact."""
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )
    else:
        if not (KAGGLE_PROJECT_DIR / ".git").is_dir():
            raise RuntimeError(
                f"Kaggle project path exists but is not a Git clone: {KAGGLE_PROJECT_DIR}"
            )
        subprocess.run(
            ["git", "-C", str(KAGGLE_PROJECT_DIR), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
    return KAGGLE_PROJECT_DIR.resolve()

def find_project_root() -> Path | None:
    direct_candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate.resolve()
    return None

PROJECT_ROOT = sync_kaggle_project() if KAGGLE else find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

project_root_string = str(PROJECT_ROOT)
while project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)

# Run All can reuse a live Kaggle kernel. Remove previously imported project
# modules so an updated working clone cannot be shadowed by stale objects.
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "project_source_policy": "working_clone_main" if KAGGLE else "local_project_root",
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Notebook tổng hợp CSV/JSON từ các notebook trước, không train model và không thay đổi threshold.
Attach outputs mới nhất của Notebook 02-07 bằng Add Input khi chạy trên Kaggle. Reference model
luôn được đọc từ frozen manifest đã chọn bằng validation; test không được dùng để chọn lại model,
calibration, threshold hoặc rule subset.

In [ ]:
from src.artifacts import (
    find_result_file, load_frozen_reference_artifact, sha256_file, stable_config_hash,
)
from src.data import load_config
from src.evaluation import expected_calibration_error
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    log_loss, precision_recall_curve,
)

ieee_config_path = PROJECT_ROOT / "configs/ieee_cis.yaml"
baf_config_path = PROJECT_ROOT / "configs/baf.yaml"
ieee_config = load_config(ieee_config_path)
baf_config = load_config(baf_config_path)
ieee_artifact = load_frozen_reference_artifact("ieee_cis", expected_config=ieee_config, search_roots=INPUT_ROOTS)
baf_artifact = load_frozen_reference_artifact("baf", expected_config=baf_config, search_roots=INPUT_ROOTS)
artifacts = {"IEEE-CIS": ieee_artifact, "BAF": baf_artifact}
configs = {"IEEE-CIS": ieee_config, "BAF": baf_config}
config_paths = {"IEEE-CIS": ieee_config_path, "BAF": baf_config_path}
for dataset, frozen in artifacts.items():
    if bool(frozen["manifest"]["quick_run"]) != QUICK_RUN:
        raise ValueError(f"{dataset} frozen artifact mode does not match Notebook 08")
    if int(frozen["manifest"]["reference_seed"]) != int(configs[dataset]["evaluation"]["reference_seed"]):
        raise ValueError(f"{dataset} frozen artifact reference seed violates the locked protocol")
    if not QUICK_RUN and str(frozen["manifest"].get("data_source", "")).lower() == "synthetic":
        raise ValueError(f"{dataset} uses a synthetic-fallback artifact and cannot support final thesis claims")

ieee_benchmark_root = ieee_artifact["manifest_path"].parent
baf_benchmark_root = baf_artifact["manifest_path"].parent
output_dir = OUTPUT_BASE / "08_cross_dataset_result_synthesis"
output_dir.mkdir(parents=True, exist_ok=True)

required = {
    "ieee_predictive": ieee_benchmark_root / "predictive_metrics_summary.csv",
    "baf_predictive": baf_benchmark_root / "predictive_metrics_summary.csv",
    "ieee_bootstrap": ieee_benchmark_root / "paired_bootstrap_model_differences.csv",
    "baf_bootstrap": baf_benchmark_root / "paired_bootstrap_model_differences.csv",
    "ieee_rules": find_result_file("ieee_rule_quality.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_satisfaction": find_result_file("ieee_knowledge_base_satisfaction.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_stability": find_result_file("ieee_rule_stability.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_thresholds": find_result_file("ieee_fitted_rule_thresholds.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_rationale": find_result_file("ieee_rule_rationale.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_rule_lineage": find_result_file("upstream_lineage.json", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_explanations": find_result_file("ieee_explanation_quality.csv", INPUT_ROOTS, "05_ieee_cis_rule_explanation_evaluation"),
    "ieee_explanation_lineage": find_result_file("upstream_lineage.json", INPUT_ROOTS, "05_ieee_cis_rule_explanation_evaluation"),
    "ieee_ablation": find_result_file("ieee_rule_ablation.csv", INPUT_ROOTS, "06_ieee_cis_rule_ablation"),
    "ieee_ablation_lineage": find_result_file("upstream_lineage.json", INPUT_ROOTS, "06_ieee_cis_rule_ablation"),
    "baf_rules": find_result_file("baf_rule_quality.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_explanations": find_result_file("baf_explanation_quality.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_satisfaction": find_result_file("baf_knowledge_base_satisfaction.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_stability": find_result_file("baf_rule_stability.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_thresholds": find_result_file("baf_fitted_rule_thresholds.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_rationale": find_result_file("baf_rule_rationale.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_replication_lineage": find_result_file("upstream_lineage.json", INPUT_ROOTS, "07_baf_ltn_generalization"),
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required upstream outputs are missing: {missing}")

manifest_items = {
    **required,
    "ieee_frozen_manifest": ieee_artifact["manifest_path"],
    "ieee_frozen_artifact": ieee_artifact["artifact_path"],
    "baf_frozen_manifest": baf_artifact["manifest_path"],
    "baf_frozen_artifact": baf_artifact["artifact_path"],
}
input_manifest = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }
    for name, path in manifest_items.items()
])
display(input_manifest)
input_manifest.to_csv(output_dir / "input_artifact_manifest.csv", index=False)

lineage_specs = [
    {
        "dataset": "IEEE-CIS", "stage": "04", "path": required["ieee_rule_lineage"],
        "notebook_id": "04_IEEE_CIS_LTN_Rule_Analysis", "frozen": None,
    },
    {
        "dataset": "IEEE-CIS", "stage": "05", "path": required["ieee_explanation_lineage"],
        "notebook_id": "05_IEEE_CIS_Rule_Explanation_Evaluation", "frozen": ieee_artifact,
    },
    {
        "dataset": "IEEE-CIS", "stage": "06", "path": required["ieee_ablation_lineage"],
        "notebook_id": "06_IEEE_CIS_Rule_Ablation", "frozen": ieee_artifact,
    },
    {
        "dataset": "BAF", "stage": "07", "path": required["baf_replication_lineage"],
        "notebook_id": "07_BAF_Cross_Dataset_Rule_and_Explanation_Replication", "frozen": baf_artifact,
    },
]
lineage_rows = []
for spec in lineage_specs:
    dataset = spec["dataset"]
    lineage_path = spec["path"]
    frozen = spec["frozen"]
    payload = json.loads(lineage_path.read_text(encoding="utf-8"))
    output_files = list(payload.get("output_files", []))
    output_hashes = dict(payload.get("output_sha256", {}))
    outputs_present = bool(output_files) and all(
        (lineage_path.parent / name).is_file() for name in output_files
    )
    outputs_hash_bound = (
        outputs_present
        and set(output_files) == set(output_hashes)
        and all(
            sha256_file(lineage_path.parent / name) == output_hashes[name]
            for name in output_files
        )
    )
    checks = {
        "notebook_match": payload.get("notebook_id") == spec["notebook_id"],
        "dataset_match": str(payload.get("dataset_name", "")).lower() == str(configs[dataset]["dataset"]["name"]).lower(),
        "config_match": payload.get("config_sha256") == stable_config_hash(configs[dataset]),
        "source_fingerprint_match": payload.get("audit_source_sha256") == audit_pipeline_fingerprint(config_paths[dataset]),
        "manifest_match": (
            True if frozen is None else
            payload.get("frozen_manifest_sha256") == sha256_file(frozen["manifest_path"])
        ),
        "artifact_match": (
            True if frozen is None else
            payload.get("frozen_artifact_sha256") == sha256_file(frozen["artifact_path"])
        ),
        "reference_seed_match": (
            True if frozen is None else
            int(payload.get("reference_seed", -1)) == int(configs[dataset]["evaluation"]["reference_seed"])
        ),
        "mode_match": bool(payload.get("quick_run")) == QUICK_RUN,
        "data_source_acceptable": QUICK_RUN or str(payload.get("data_source", "")).lower() != "synthetic",
        "declared_outputs_present": outputs_present,
        "output_hashes_match": outputs_hash_bound,
    }
    lineage_rows.append({
        "dataset": dataset, "stage": spec["stage"],
        "git_commit": payload.get("git_commit"), "lineage": str(lineage_path), **checks,
    })
lineage_audit = pd.DataFrame(lineage_rows)
display(lineage_audit)
audit_checks = lineage_audit.drop(columns=["dataset", "stage", "git_commit", "lineage"])
if not audit_checks.all(axis=None):
    raise ValueError("At least one Notebook 04-07 output failed provenance or alignment checks")
lineage_audit.to_csv(output_dir / "upstream_lineage_audit.csv", index=False)

## Predictive evidence

In [ ]:
predictive = pd.concat([
    pd.read_csv(required["ieee_predictive"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_predictive"]).assign(dataset="BAF"),
], ignore_index=True)
predictive_test = predictive.query("split == 'test'").copy()
reference_keys = {
    "IEEE-CIS": ieee_artifact["manifest"]["model_key"],
    "BAF": baf_artifact["manifest"]["model_key"],
}
reference_rows = pd.concat([
    predictive.loc[
        (predictive["dataset"] == dataset)
        & (predictive["model_key"] == model_key)
        & (predictive["split"] == "test")
    ]
    for dataset, model_key in reference_keys.items()
], ignore_index=True)
if len(reference_rows) != len(reference_keys):
    raise ValueError("Frozen reference model rows are missing or duplicated in predictive summaries")
prevalence = {
    dataset: float(frozen["y_test"].mean()) for dataset, frozen in artifacts.items()
}
reference_rows["test_prevalence"] = reference_rows["dataset"].map(prevalence)
reference_rows["pr_auc_over_prevalence"] = (
    reference_rows["raw_pr_auc_mean"] / reference_rows["test_prevalence"]
)

reference_split = predictive.loc[
    predictive.apply(lambda row: row["model_key"] == reference_keys[row["dataset"]], axis=1)
].copy()
reference_gap = reference_split.pivot(
    index=["dataset", "model", "model_key"], columns="split", values="raw_pr_auc_mean"
).reset_index()
reference_gap["validation_to_test_gap"] = reference_gap["test"] - reference_gap["validation"]

display(predictive_test.round(4), reference_rows.round(4), reference_gap.round(4))
predictive_test.to_csv(output_dir / "table_cross_dataset_predictive_test.csv", index=False)
reference_rows.to_csv(output_dir / "table_frozen_reference_predictive_test.csv", index=False)
reference_gap.to_csv(output_dir / "table_validation_to_test_gap.csv", index=False)

bootstrap = pd.concat([
    pd.read_csv(required["ieee_bootstrap"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_bootstrap"]).assign(dataset="BAF"),
], ignore_index=True)
bootstrap["uncertainty_scope"] = "paired row-level bootstrap on frozen reference-seed predictions"
display(bootstrap.round(5))
bootstrap.to_csv(output_dir / "table_cross_dataset_predictive_bootstrap.csv", index=False)

confusion_rows = []
calibration_rows = []
for dataset, frozen in artifacts.items():
    labels = frozen["y_test"].astype(int)
    raw_probability = frozen["test_raw_probability"].astype(float)
    calibrated_probability = frozen["test_probability"].astype(float)
    decision_threshold = float(frozen["manifest"]["threshold"])
    tn, fp, fn, tp = confusion_matrix(
        labels, calibrated_probability >= decision_threshold, labels=[0, 1]
    ).ravel()
    confusion_rows.append({
        "dataset": dataset, "threshold": decision_threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_alert_count": int(fp + tp), "test_rows": len(labels),
    })
    for probability_type, probability in (
        ("raw", raw_probability), ("calibrated", calibrated_probability)
    ):
        calibration_rows.append({
            "dataset": dataset,
            "probability_type": probability_type,
            "pr_auc": average_precision_score(labels, probability),
            "brier": brier_score_loss(labels, probability),
            "ece": expected_calibration_error(labels, probability, n_bins=15),
            "nll": log_loss(labels, np.clip(probability, 1e-7, 1 - 1e-7), labels=[0, 1]),
        })
confusion_table = pd.DataFrame(confusion_rows)
calibration_table = pd.DataFrame(calibration_rows)
display(confusion_table, calibration_table.round(5))
confusion_table.to_csv(output_dir / "table_frozen_reference_confusion_counts.csv", index=False)
calibration_table.to_csv(output_dir / "table_raw_vs_calibrated_probability_metrics.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=False)
model_color = "#4C72B0"
for axis, dataset in zip(axes, ("IEEE-CIS", "BAF")):
    subset = predictive_test.loc[predictive_test["dataset"] == dataset].sort_values("raw_pr_auc_mean")
    positions = np.arange(len(subset))
    axis.bar(
        positions, subset["raw_pr_auc_mean"],
        yerr=subset["raw_pr_auc_std"].fillna(0),
        color=model_color, edgecolor="#2F4B66", capsize=4,
    )
    axis.axhline(prevalence[dataset], color="#4A4A4A", linestyle="--", linewidth=1.2,
                 label=f"Test prevalence = {prevalence[dataset]:.4f}")
    axis.set_xticks(positions, subset["model"], rotation=18, ha="right")
    axis.set_title(f"{dataset} test raw PR-AUC")
    axis.set_ylabel("Raw PR-AUC (mean ± SD across 3 seeds)")
    axis.legend(loc="upper left")
    axis.margins(x=0.10)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.22, top=0.88, wspace=0.26)
fig.savefig(output_dir / "figure_predictive_performance_by_dataset.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16.5, 5.4))
for axis, (dataset, frozen) in zip(axes, artifacts.items()):
    labels = frozen["y_test"].astype(int)
    raw_probability = frozen["test_raw_probability"].astype(float)
    calibrated_probability = frozen["test_probability"].astype(float)
    for name, probability, color, line_style in (
        ("Raw score", raw_probability, "#4C72B0", "-"),
        ("Calibrated probability", calibrated_probability, "#DD8452", "--"),
    ):
        precision, recall, _ = precision_recall_curve(labels, probability)
        axis.plot(recall, precision, color=color, linestyle=line_style, linewidth=2, label=name)
    axis.axhline(prevalence[dataset], color="#4A4A4A", linestyle=":", label="Prevalence")
    axis.set_title(f"{dataset} frozen-reference precision-recall curve")
    axis.set_xlabel("Recall")
    axis.set_ylabel("Precision")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.legend(loc="upper right")
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.88, wspace=0.38)
fig.savefig(output_dir / "figure_frozen_reference_pr_curves.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16.5, 5.4))
for axis, (dataset, frozen) in zip(axes, artifacts.items()):
    labels = frozen["y_test"].astype(int)
    for name, probability, color, marker in (
        ("Raw score", frozen["test_raw_probability"], "#4C72B0", "o"),
        ("Calibrated probability", frozen["test_probability"], "#DD8452", "s"),
    ):
        observed, predicted = calibration_curve(labels, probability, n_bins=10, strategy="quantile")
        axis.plot(predicted, observed, color=color, marker=marker, linewidth=1.8, label=name)
    axis.plot([0, 1], [0, 1], color="#4A4A4A", linestyle="--", linewidth=1, label="Ideal")
    axis.set_title(f"{dataset} reliability diagram")
    axis.set_xlabel("Mean predicted probability")
    axis.set_ylabel("Observed fraud rate")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.legend(loc="upper left")
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.88, wspace=0.38)
fig.savefig(output_dir / "figure_frozen_reference_reliability.png", dpi=180, bbox_inches="tight")
plt.show()

## Rule, explanation and ablation evidence

In [ ]:
rules = pd.concat([
    pd.read_csv(required["ieee_rules"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_rules"]).assign(dataset="BAF"),
], ignore_index=True)
explanations = pd.concat([
    pd.read_csv(required["ieee_explanations"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_explanations"]).assign(dataset="BAF"),
], ignore_index=True)
satisfaction = pd.concat([
    pd.read_csv(required["ieee_satisfaction"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_satisfaction"]).assign(dataset="BAF"),
], ignore_index=True)
stability = pd.concat([
    pd.read_csv(required["ieee_stability"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_stability"]).assign(dataset="BAF"),
], ignore_index=True)
fitted_thresholds = pd.concat([
    pd.read_csv(required["ieee_thresholds"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_thresholds"]).assign(dataset="BAF"),
], ignore_index=True)
rule_rationale = pd.concat([
    pd.read_csv(required["ieee_rationale"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_rationale"]).assign(dataset="BAF"),
], ignore_index=True)
ablation = pd.read_csv(required["ieee_ablation"])
display(
    rules.round(4), satisfaction.round(4), stability.round(4),
    explanations.round(4), ablation.round(4), fitted_thresholds, rule_rationale,
)
rules.to_csv(output_dir / "table_cross_dataset_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "table_cross_dataset_knowledge_base_satisfaction.csv", index=False)
stability.to_csv(output_dir / "table_cross_dataset_rule_stability.csv", index=False)
explanations.to_csv(output_dir / "table_cross_dataset_explanation_quality.csv", index=False)
ablation.to_csv(output_dir / "table_ieee_rule_ablation.csv", index=False)
fitted_thresholds.to_csv(output_dir / "table_cross_dataset_fitted_rule_thresholds.csv", index=False)
rule_rationale.to_csv(output_dir / "table_cross_dataset_rule_rationale.csv", index=False)

In [ ]:
test_rules = rules.query("split == 'test'") if "split" in rules else rules
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for axis, dataset in zip(axes, ("IEEE-CIS", "BAF")):
    subset = (
        test_rules.loc[(test_rules["dataset"] == dataset) & test_rules["lift"].notna()]
        .query("active_count > 0")
        .sort_values("lift")
    )
    positions = np.arange(len(subset))
    axis.barh(positions, subset["lift"], color="#4C72B0", edgecolor="#2F4B66")
    axis.set_yticks(positions, subset["rule"])
    axis.axvline(1.0, color="#4A4A4A", linestyle="--", linewidth=1.2)
    axis.set_title(f"{dataset} test rule lift with denominators")
    axis.set_xlabel("Fraud-rate lift over test prevalence")
    axis.set_ylabel("Rule")
    axis.set_xlim(0, max(1.2, float(subset["lift"].max()) * 1.32))
    for position, (_, row) in enumerate(subset.iterrows()):
        axis.text(
            row["lift"] + float(subset["lift"].max()) * 0.025,
            position,
            f"n={int(row['active_count'])}; coverage={row['coverage']:.2%}",
            va="center", fontsize=9,
        )
fig.subplots_adjust(left=0.15, right=0.98, bottom=0.11, top=0.90, wspace=0.72)
fig.savefig(output_dir / "figure_rule_lift_with_denominators.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
explanation_plot = explanations.copy()
x = np.arange(len(explanation_plot))
width = 0.34
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
axes[0].bar(
    x - width / 2, explanation_plot["all_alert_precision"], width,
    label="All predicted alerts", color="#9CB8D2", edgecolor="#2F4B66",
)
axes[0].bar(
    x + width / 2, explanation_plot["explained_alert_precision"], width,
    label="Alerts with rule evidence", color="#4C72B0", edgecolor="#2F4B66",
)
axes[0].set_xticks(x, explanation_plot["dataset"])
axes[0].set_ylabel("Fraud precision")
axes[0].set_title("Alert precision with and without rule-evidence filtering")
axes[0].legend(loc="upper right")

gain = explanation_plot["explained_alert_precision_gain"].to_numpy(float)
ci_low = explanation_plot["precision_gain_ci_low"].to_numpy(float)
ci_high = explanation_plot["precision_gain_ci_high"].to_numpy(float)
for position, (point, low, high) in enumerate(zip(gain, ci_low, ci_high)):
    if np.isfinite([point, low, high]).all():
        axes[1].hlines(position, low, high, color="#8C5A3C", linewidth=2)
        axes[1].plot(point, position, "o", markersize=8, color="#DD8452")
        axes[1].vlines([low, high], position - 0.08, position + 0.08, color="#8C5A3C")
    else:
        axes[1].text(0, position, "undefined", ha="center", va="center", fontsize=9)
axes[1].axvline(0, color="#4A4A4A", linestyle="--", linewidth=1)
axes[1].set_yticks(x, explanation_plot["dataset"])
axes[1].set_xlabel("Explained-alert precision gain (95% bootstrap CI)")
axes[1].set_title("Selective explanation precision gain")
for position, (_, row) in enumerate(explanation_plot.iterrows()):
    if np.isfinite(row["precision_gain_ci_high"]):
        axes[1].annotate(
            f"alert coverage={row['explanation_coverage_alerts']:.1%}; n={int(row['explained_alert_count'])}",
            (row["precision_gain_ci_high"], position), xytext=(8, 0),
            textcoords="offset points", va="center", fontsize=9,
        )
fig.subplots_adjust(left=0.10, right=0.95, bottom=0.14, top=0.88, wspace=0.38)
fig.savefig(output_dir / "figure_cross_dataset_explanation_evidence.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
diagnostic = ablation.dropna(subset=["coverage_alerts", "precision_gain"]).copy()
diagnostic["family"] = np.select(
    [
        diagnostic["ablation"].str.startswith("only:"),
        diagnostic["ablation"].str.startswith("without:"),
        diagnostic["ablation"].eq("full_rule_set"),
        diagnostic["ablation"].eq("full_rule_set_sensitivity"),
    ],
    ["only one rule", "leave one out", "full set", "activation sensitivity"],
    default="other",
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))
palette = {
    "only one rule": "#4C72B0", "leave one out": "#DD8452",
    "full set": "#2F4B66", "activation sensitivity": "#9B8F45",
}
subset_diagnostics = diagnostic.loc[diagnostic["family"].isin(["only one rule", "leave one out", "full set"])]
for family, group in subset_diagnostics.groupby("family"):
    axes[0].scatter(
        group["coverage_alerts"], group["precision_gain"],
        label=family, color=palette[family], s=60, alpha=0.85,
    )
full = diagnostic.loc[diagnostic["ablation"] == "full_rule_set"].iloc[0]
axes[0].annotate("full rule set", (full["coverage_alerts"], full["precision_gain"]),
                 xytext=(8, 8), textcoords="offset points")
axes[0].axhline(0, color="#4A4A4A", linestyle="--", linewidth=1)
axes[0].set_xlabel("Alert explanation coverage")
axes[0].set_ylabel("Explained-alert precision gain")
axes[0].set_title("IEEE-CIS post-hoc rule-subset diagnostics")
axes[0].legend(loc="best")

sensitivity = diagnostic.loc[diagnostic["family"] == "activation sensitivity"].sort_values("activation_threshold")
axes[1].plot(
    sensitivity["coverage_alerts"], sensitivity["precision_gain"],
    marker="o", color=palette["activation sensitivity"], linewidth=2,
)
label_offsets = [(7, 8), (7, 20), (7, -14), (7, -26)]
for label_index, (_, row) in enumerate(sensitivity.iterrows()):
    axes[1].annotate(
        f"t={row['activation_threshold']:.2f}",
        (row["coverage_alerts"], row["precision_gain"]),
        xytext=label_offsets[label_index % len(label_offsets)],
        textcoords="offset points", fontsize=9,
    )
axes[1].axhline(0, color="#4A4A4A", linestyle="--", linewidth=1)
axes[1].set_xlabel("Alert explanation coverage")
axes[1].set_ylabel("Explained-alert precision gain")
axes[1].set_title("IEEE-CIS activation-threshold sensitivity (post-hoc)")
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.14, top=0.88, wspace=0.26)
fig.savefig(output_dir / "figure_ieee_ablation_coverage_precision_tradeoff.png", dpi=180, bbox_inches="tight")
plt.show()

## Thesis evidence matrix

In [ ]:
ieee_explanation = explanations.loc[explanations["dataset"] == "IEEE-CIS"].iloc[0]
baf_explanation = explanations.loc[explanations["dataset"] == "BAF"].iloc[0]
ieee_reference = reference_rows.loc[reference_rows["dataset"] == "IEEE-CIS"].iloc[0]
baf_reference = reference_rows.loc[reference_rows["dataset"] == "BAF"].iloc[0]

def enrichment_status(row):
    values = row[[
        "explained_alert_precision_gain", "precision_gain_ci_low", "precision_gain_ci_high",
    ]].to_numpy(float)
    if not np.isfinite(values).all():
        return "undefined"
    if float(row["precision_gain_ci_low"]) > 0:
        return "positive"
    if float(row["precision_gain_ci_high"]) < 0:
        return "negative"
    return "inconclusive (CI includes zero)"

enrichment = {
    "IEEE-CIS": enrichment_status(ieee_explanation),
    "BAF": enrichment_status(baf_explanation),
}
positive_datasets = [name for name, status in enrichment.items() if status == "positive"]
if len(positive_datasets) == len(enrichment):
    rq3_supported_claim = (
        "Positive alert enrichment from rule-evidence filtering is supported on both evaluated benchmarks."
    )
elif positive_datasets:
    remaining = [name for name in enrichment if name not in positive_datasets]
    rq3_supported_claim = (
        f"Positive enrichment is supported for {', '.join(positive_datasets)}, but cross-dataset "
        f"replication is not established because {', '.join(remaining)} is "
        f"{', '.join(enrichment[name] for name in remaining)}."
    )
else:
    status_text = "; ".join(f"{name}: {status}" for name, status in enrichment.items())
    rq3_supported_claim = (
        f"The current intervals do not establish positive alert enrichment on either benchmark ({status_text})."
    )
evidence_matrix = pd.DataFrame([
    {
        "research_question": "RQ1 - Leakage-aware fraud prediction",
        "observed_evidence": (
            f"IEEE {ieee_reference['model']} raw PR-AUC={ieee_reference['raw_pr_auc_mean']:.4f}; "
            f"BAF {baf_reference['model']} raw PR-AUC={baf_reference['raw_pr_auc_mean']:.4f}."
        ),
        "supported_claim": "The locked temporal protocol yields traceable predictive measurements and validation-selected frozen references on both benchmarks.",
        "claim_boundary": "No SOTA, universal superiority, or production-readiness claim.",
    },
    {
        "research_question": "RQ2 - Fuzzy knowledge representation",
        "observed_evidence": "Train-fitted fuzzy rules, fitted thresholds, per-rule coverage/lift and balanced KB satisfaction are reported.",
        "supported_claim": "Domain/data-informed hypotheses can be represented as an auditable fuzzy-rule layer.",
        "claim_boundary": "LTN-inspired diagnostic, not an end-to-end co-trained LTN predictor.",
    },
    {
        "research_question": "RQ3 - Selective alert audit value",
        "observed_evidence": (
            f"Precision gain: IEEE {ieee_explanation['explained_alert_precision_gain']:.3f} "
            f"(CI {ieee_explanation['precision_gain_ci_low']:.3f} to {ieee_explanation['precision_gain_ci_high']:.3f}); "
            f"BAF {baf_explanation['explained_alert_precision_gain']:.3f} "
            f"(CI {baf_explanation['precision_gain_ci_low']:.3f} to {baf_explanation['precision_gain_ci_high']:.3f})."
        ),
        "supported_claim": rq3_supported_claim,
        "claim_boundary": "Association/enrichment, not causal or model-faithful explanation.",
    },
    {
        "research_question": "RQ4 - Coverage-precision trade-off",
        "observed_evidence": "Rule-subset and activation-threshold diagnostics expose changes in alert coverage and explained-alert precision.",
        "supported_claim": "Post-hoc diagnostics quantify how explanation coverage and precision gain vary across rule subsets and activation thresholds.",
        "claim_boundary": "Locked-test ablation is post-hoc and cannot select a new unbiased final rule set.",
    },
    {
        "research_question": "RQ5 - Cross-dataset applicability and reproducibility",
        "observed_evidence": "Frozen checksums, split alignment, lineage manifests and BAF-specific replication are verified.",
        "supported_claim": "The same audited pipeline is reproducibly executed on a second benchmark with dataset-specific rules and models.",
        "claim_boundary": "Not transfer of the same model/rules and not external institutional validation.",
    },
])
display(evidence_matrix)
evidence_matrix.to_csv(output_dir / "table_thesis_evidence_matrix.csv", index=False)

## Takeaways

In [ ]:
display(reference_rows[[
    "dataset", "model", "raw_pr_auc_mean", "raw_pr_auc_std",
    "test_prevalence", "pr_auc_over_prevalence", "fbeta_mean", "brier_mean", "ece_mean",
]].round(4))
display(Markdown(
    "- Reference models above come from frozen manifests selected on validation; Notebook 08 never reselects from test.\n"
    "- Raw PR-AUC supports ranking; calibrated probabilities support Brier/ECE/NLL and locked-threshold decisions.\n"
    "- The paired bootstrap is row-level uncertainty for the frozen reference seed, while three-seed SD is descriptive run variability.\n"
    f"- Rule-evidence result: {rq3_supported_claim}\n"
    "- BAF supports framework replication with BAF-specific rules, not model/rule transfer, causal explanation, or production generalization."
))